<a href="https://colab.research.google.com/github/AhmedMahmoud-123/FlyRank_AI/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
import os

REPO_DIR = '/content/FlyRank_AI'

if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/AhmedMahmoud-123/FlyRank_AI.git

os.chdir(REPO_DIR)

!python scripts/01_prepare_features.py

import pandas as pd

df = pd.read_csv('data/processed/refresh_feature_vector.csv')
print(f'{len(df):,} rows loaded')

Prepared 30,000 rows from 30,000 raw rows
Wrote /content/FlyRank_AI/data/processed/refresh_feature_vector.csv
30,000 rows loaded


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method:** Logistic Regression first, then Random Forest.

My lane predicts `is_declining_label` — a yes/no outcome with an observed label — which per
the toolkit's question-shape table means "start readable, then go stronger." Logistic
Regression gives interpretable coefficients I can sanity-check against domain intuition
before trusting anything fancier. Random Forest is the natural next step because it can
pick up non-linear interactions (e.g. staleness only matters *combined with* prior
visibility) that a linear model can't. I'm skipping Gradient Boosting for now — the ML-07
baseline already gets meaningful lift from simple rules, so added complexity needs to earn
its place per training-honest-models, not be assumed.

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

feature_cols = [
    'impressions_prev_30d',
    'avg_position',
    'days_since_last_update',
    'has_clicks'
]

X = df[feature_cols]
y = df['is_declining_label']

print('Features used:')
print(feature_cols)

print('\nMissing values:')
print(X.isna().sum())

Features used:
['impressions_prev_30d', 'avg_position', 'days_since_last_update', 'has_clicks']

Missing values:
impressions_prev_30d      0
avg_position              0
days_since_last_update    0
has_clicks                0
dtype: int64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split:** GroupShuffleSplit on `client_id`, not a random row split.

Per the data dictionary, `client_id` is a pseudonym for grouping only, and content items
from the same client share client-level patterns (their typical position range, their
update cadence). A random split lets those patterns leak across train/test, inflating the
score in a way that won't hold up on a client the model has never seen — the same leakage
notebook 02 demonstrated. GroupShuffleSplit keeps every row from one client entirely on one
side of the split, matching the honest-validation approach from `hunting-leakage-and-validating`.

In [18]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['client_id']))
X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

print('clients in train:', df.iloc[train_idx]['client_id'].nunique())
print('clients in test: ', df.iloc[test_idx]['client_id'].nunique())
print('base rate (test):', y_te.mean().round(3))

clients in train: 24
clients in test:  8
base rate (test): 0.517


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Comparison:** same test split (by client), same metric (precision@K), same base rate
reference as the Week-4 baseline. The baseline's ranked queue is recomputed here on the
*test* rows only, so it's a fair head-to-head, not a different data slice.

In [19]:
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# --- baseline (Week 4 rule), recomputed on the test split only ---
test_df = df.iloc[test_idx].copy()

visible = (test_df['impressions_prev_30d'] >= 500).astype(int)
stale = (test_df['days_since_last_update'] >= 180).astype(int)
slipping = (
    test_df['avg_position'] >
    test_df.loc[test_df['avg_position'] > 0, 'avg_position'].median()
).astype(int)

baseline_score = (
    visible * (stale + slipping) * test_df['impressions_prev_30d']
)

# --- logistic regression ---
logreg = LogisticRegression(
    max_iter=1000,
    random_state=42
).fit(X_tr, y_tr)

logreg_score = logreg.predict_proba(X_te)[:, 1]

# --- random forest ---
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
).fit(X_tr, y_tr)

rf_score = rf.predict_proba(X_te)[:, 1]

results = []

for name, scores in [
    ('baseline_rule', baseline_score),
    ('logistic_regression', logreg_score),
    ('random_forest', rf_score)
]:
    row = {'model': name}

    for k in [20, 50, 100]:
        row[f'precision@{k}'] = round(
            precision_at_k(scores, y_te.values, k), 3
        )

    results.append(row)

comparison = pd.DataFrame(results)
comparison['base_rate'] = round(y_te.mean(), 3)

comparison

,model,precision@20,precision@50,precision@100,base_rate
0,baseline_rule,0.55,0.58,0.51,0.517
1,logistic_regression,0.70,0.74,0.71,0.517
2,random_forest,0.55,0.60,0.70,0.517


In [20]:
# Compare each model against the baseline

baseline = comparison.loc[
    comparison['model'] == 'baseline_rule'
].iloc[0]

for model_name in ['logistic_regression', 'random_forest']:
    row = comparison.loc[
        comparison['model'] == model_name
    ].iloc[0]

    print(f'\n{model_name}')

    for k in [20, 50, 100]:
        improvement = row[f'precision@{k}'] - baseline[f'precision@{k}']
        print(f'P@{k} improvement over baseline: {improvement:+.3f}')


logistic_regression
P@20 improvement over baseline: +0.150
P@50 improvement over baseline: +0.160
P@100 improvement over baseline: +0.200

random_forest
P@20 improvement over baseline: +0.000
P@50 improvement over baseline: +0.020
P@100 improvement over baseline: +0.190


### How to read Precision@K

Precision@K measures how many of the first K ranked items are actually labeled as declining.

For example, a Precision@20 of 0.70 means that 14 of the top 20 ranked items were labeled as declining.

The values are calculated separately for each K, so Precision@K does not have to decrease as K increases.

### Model comparison

Logistic Regression performs best at the smaller review sizes, with **70% Precision@20** and **74% Precision@50**. Random Forest reaches **70% Precision@100**, but is less precise than Logistic Regression at the smaller K values.

The baseline reaches **55% Precision@20**, **58% Precision@50**, and **51% Precision@100**. Both machine-learning models improve on the baseline for at least some review sizes.

These results are from one client-grouped test split, so I treat them as a model comparison rather than the final estimate of generalization. W06 will provide the final grouped validation.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [21]:
# Feature importance for Random Forest
# Random Forest is included because its tree-based feature importance
# helps show which inputs it relies on.
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances)

# permutation importance as a cross-check (importances above can overstate high-cardinality features)
from sklearn.inspection import permutation_importance
perm = permutation_importance(rf, X_te, y_te, n_repeats=10, random_state=42, n_jobs=-1)
pd.Series(perm.importances_mean, index=feature_cols).sort_values(ascending=False)

impressions_prev_30d      0.553510
avg_position              0.362688
days_since_last_update    0.071142
has_clicks                0.012660
dtype: float64


,0
impressions_prev_30d,0.106788
avg_position,0.017372
has_clicks,0.016824
days_since_last_update,0.014434


### What the model is using

The Random Forest relies most heavily on `impressions_prev_30d` and `avg_position`. Their feature importances are approximately **0.554** and **0.363**, respectively. `days_since_last_update` contributes much less, while `has_clicks` has only a small contribution.

This is consistent with the task: recent visibility and search position contain most of the predictive signal available before the prediction window.

The permutation-importance check also ranks `impressions_prev_30d` as the strongest feature, followed by `avg_position`. This provides a second view of which inputs the Random Forest relies on. I treat these results as model associations, not causal effects.

In [24]:
# Use 0.5 only for simple error analysis.
# The intended workflow ranks items for review rather than using
# a fixed production classification threshold.
test_df['rf_pred'] = (rf_score >= 0.5).astype(int)

wrong = test_df[
    test_df['rf_pred'] != test_df['is_declining_label']
]

wrong[
    [
        'content_id',
        'impressions_prev_30d',
        'avg_position',
        'days_since_last_update',
        'is_declining_label',
        'rf_score'
    ]
].sample(3, random_state=42)

,content_id,impressions_prev_30d,avg_position,days_since_last_update,is_declining_label,rf_score
9788,content_13f687a89060,80,6.0,20,0,0.575
26866,content_b12f1322a7d4,13,16.4,20,0,0.930
4987,content_809b50c43cad,140,29.7,13,0,0.760


### False-positive examples

The sampled false positives show that the Random Forest can assign high decline-risk scores to pages that are not labeled as declining.

For example, one page received a score of **0.930** despite having a non-declining label. Other false positives received scores of **0.760** and **0.575**.

These examples show why the model should be used to prioritize pages for human review rather than automatically classify pages as declining. A high model score indicates that a page should be investigated, not that decline is certain.

## Interpretation

After removing the two features flagged for possible temporal overlap, Logistic Regression gives the strongest Precision@20 and Precision@50 on the client-grouped test split, while Random Forest performs similarly at larger review capacity.

The strongest Random Forest signals are `impressions_prev_30d` and `avg_position`. These describe recent visibility and search position before the prediction window, so they are more appropriate for a forward-looking review workflow.

These models should therefore be treated as ranking tools for prioritizing human review, not as automated decision rules or causal explanations of traffic decline.

The grouped validation in W06 is still required before treating these test-set results as the final performance estimate.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.